In [ ]:
Names = "Diane Umuhoza"
ID = 101286

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from ISLP import load_data
from sklearn.model_selection import train_test_split, cross_val_score, KFold, LeaveOneOut
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, accuracy_score
from sklearn.utils import resample

In [ ]:
Auto = load_data("Auto").dropna()
X = Auto[["horsepower"]]
y = Auto["mpg"]
for seed in [1, 2, 3]:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=seed)
    model = LinearRegression().fit(X_train, y_train)
    print({"Seed": seed, "Test_MSE": mean_squared_error(y_test, model.predict(X_test))})

In [ ]:
cv = KFold(n_splits=10, shuffle=True, random_state=1)
scores = -cross_val_score(LinearRegression(), X, y, cv=cv, scoring="neg_mean_squared_error")
print({"10_fold_CV_MSE": scores.mean(), "Fold_MSE": scores})

In [ ]:
Default = load_data("Default")
X = Default[["income", "balance"]]
y = (Default["default"] == "Yes").astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=1, stratify=y)
logit = LogisticRegression(max_iter=2000).fit(X_train, y_train)
print({"Validation_Error": 1 - accuracy_score(y_test, logit.predict(X_test))})

In [ ]:
rng = np.random.default_rng(1)
coefficients = []
for _ in range(1000):
    indices = rng.integers(0, len(Default), len(Default))
    sample = Default.iloc[indices]
    model = LogisticRegression(max_iter=2000).fit(sample[["income", "balance"]], (sample["default"] == "Yes").astype(int))
    coefficients.append(model.coef_[0])
coefficients = np.asarray(coefficients)
print(pd.DataFrame(coefficients, columns=["income", "balance"]).std())